In [13]:
import torch
from transformers import BertTokenizer
import os 
import urllib.request
import tarfile
from collections import Counter
from bs4 import BeautifulSoup

In [5]:
sentences = [
    'Today is a sunny day',
    'Today is a rainy day'
]

#Tokenization function
def tokenize(text):
    return text.lower().split()

#build the vocabulary
def build_vocab(sentences):
    vocab = {}
    for sentence in sentences:
        tokens = tokenize(sentence)
        for token in tokens:
            if token not in vocab:
                vocab[token] = len(vocab)+1
    return vocab

#create the vocabulary index
vocab = build_vocab(sentences)

print("Vocabulary Index: ", vocab)

Vocabulary Index:  {'today': 1, 'is': 2, 'a': 3, 'sunny': 4, 'day': 5, 'rainy': 6}


In [7]:
#initialize the tokenizer
#BERT meaning bidirectional encoder representation from transformers
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

#tokenize the sentences and encode them
#return 'return_tensors=pt' returns the values will be torch.Tensor objects
encoded_inputs = tokenizer(sentences, padding=True,truncation=True, return_tensors='pt')

#to see the tokens for each input 
tokens = [tokenizer.convert_ids_to_tokens(ids)
          for ids in encoded_inputs["input_ids"]]

#to get the word index similar to kera's tokenizer
word_index = tokenizer.get_vocab()

print("Tokens: ", tokens)
print("Token IDs: ", encoded_inputs['input_ids'])
print("Word Index: ", dict(list(word_index.items())[:10]))
#show only 10 

Tokens:  [['[CLS]', 'today', 'is', 'a', 'sunny', 'day', '[SEP]'], ['[CLS]', 'today', 'is', 'a', 'rainy', 'day', '[SEP]']]
Token IDs:  tensor([[  101,  2651,  2003,  1037, 11559,  2154,   102],
        [  101,  2651,  2003,  1037, 16373,  2154,   102]])
Word Index:  {'resentment': 20234, '##marine': 21966, 'intimidating': 24439, 'celebrate': 8439, '##der': 4063, 'cyprus': 9719, 'moderate': 8777, '[unused672]': 677, '##rut': 22134, 'chased': 13303}


In [9]:
def download_and_extract(url,destination):
    if not os.path.exists(destination):
        os.makedirs(destination, exist_ok=True)
    file_path = os.path.join(destination, "aclImdb_v1.tar.gz")
    
    if not os.path.exists(file_path):
        print("Downloading the dataset...")
        urllib.request.urlretrieve(url,file_path)
        print("Download complete.")
        
    if "aclImdb" not in os.listdir(destination):
        print("Extracting the dataset...")
        with tarfile.open(file_path, 'r:gz') as tar:
            tar.extractall(path=destination)
        print("Extraction complete.")

#URL for dataset
dataset_url = "http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
download_and_extract(dataset_url,"./data")


Download complete.
Extracting the dataset...
Extraction complete.


In [8]:
#simple tokenizer
def tokenize(text):
    return text.lower().split()

#build vocabulary
def build_vocab(path):
    counter = Counter()
    for folder in ["pos","neg"]:
        folder_path = os.path.join(path, folder)
        for filename in os.listdir(folder_path):
            with open(os.path.join(folder_path, filename), 'r',
                      encoding='utf-8') as file:
                counter.update(tokenize(file.read()))
    return {word:i+1 for i, word in enumerate(counter)} #starting index 1

vocab = build_vocab("./data/aclImdb/train/")

In [12]:
#sequencing and padding

def text_to_sequence(text, vocab):
    return [vocab.get(token, 0) for token in tokenize(text)] #0 for unknown

def pad_sequence(sequences, maxlen):
    return [seq + [0] * (maxlen - len(seq))
            if len(seq) < maxlen else seq[:maxlen] for seq in sequences]

#example of use 
text = "This is an example."
seq = text_to_sequence(text,vocab)
padded_seq = pad_sequence([seq],maxlen=256) #example = 256
print(seq)

[100, 3, 352, 14712]


In [14]:
#stop words
stopwords = ["a", "about", "above", "after", "again", "against", "all", "am", "an", "and", "any", "are", "as", "at",
             "be", "because", "been", "before", "being", "below", "between", "both", "but", "by", "could", "did", "do",
             "does", "doing", "down", "during", "each", "few", "for", "from", "further", "had", "has", "have", "having",
             "he", "hed", "hes", "her", "here", "heres", "hers", "herself", "him", "himself", "his", "how", "hows", "i",
             "id", "ill", "im", "ive", "if", "in", "into", "is", "it", "its", "itself", "lets", "me", "more", "most",
             "my", "myself", "nor", "of", "on", "once", "only", "or", "other", "ought", "our", "ours", "ourselves",
             "out", "over", "own", "same", "she", "shed", "shell", "shes", "should", "so", "some", "such", "than",
             "that", "thats", "the", "their", "theirs", "them", "themselves", "then", "there", "theres", "these", "they",
             "theyd", "theyll", "theyre", "theyve", "this", "those", "through", "to", "too", "under", "until", "up",
             "very", "was", "we", "wed", "well", "were", "weve", "what", "whats", "when", "whens", "where", "wheres",
             "which", "while", "who", "whos", "whom", "why", "whys", "with", "would", "you", "youd", "youll", "youre",
             "youve", "your", "yours", "yourself", "yourselves"]

In [ ]:
#simple tokenizer
def tokenize(text):
    soup = BeautifulSoup(text, "html.parser")
    cleaned_text = soup.get_text() #extract text from HTML
    return [word.lower() for word in cleaned_text.split() if word.lower()
            not in stopwords]

In [ ]:
#build vocabulary for IMDB data
def build_vocab(path):
    counter = Counter()
    for folder in ["pos", "neg"]:
        folder_path = os.path.join(path, folder)
        for filename in os.listdir(folder_path):
            with open(os.path.join(folder_path, filename), 'r', encoding='utf-8') as file:
                counter.update(tokenize(file.read()))
                
    #sort words by frequency in descending order
    sorted_words = sorted(counter.items(), key=lambda x: x[1], reverse=True)
    
    vocab = {word: idx + 1 for idx, (word, _) in enumerate(sorted_words)}
    vocab['<pad>']
    return vocab


In [ ]:
vocab = build_vocab("./data/aclImdb/train/")
print(vocab)